In [1]:
import transformer_lens as tl


cfg = tl.HookedTransformerConfig(
    d_model=768,
    d_head=64,
    n_heads=12,
    n_layers=2,
    n_ctx=2048,
    d_vocab=50278,
    attention_dir="causal",
    attn_only=True,  # defaults to False
    tokenizer_name="EleutherAI/gpt-neox-20b",
    seed=398,
    use_attn_result=True,
    normalization_type=None,  # defaults to "LN", i.e. layernorm with weights & biases
    positional_embedding_type="shortformer",
)

In [2]:
from huggingface_hub import hf_hub_download
import torch as t

REPO_ID = "callummcdougall/attn_only_2L_half"
FILENAME = "attn_only_2L_half.pth"
device = "cuda"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
model = tl.HookedTransformer(cfg)
pretrained_weights = t.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(pretrained_weights)

<All keys matched successfully>

In [3]:
from playground.attn_only_transformer import AttnOnlyTransformer, Config

attn_model = AttnOnlyTransformer(Config())
attn_model.load_state_dict(pretrained_weights)

<All keys matched successfully>

In [4]:
text = "We think that powerful, significantly superhuman machine intelligence. We think that powerful"

logits, cache = model.run_with_cache(text, remove_batch_dim=True)

In [5]:
cache.keys()

dict_keys(['hook_embed', 'hook_pos_embed', 'blocks.0.hook_resid_pre', 'blocks.0.attn.hook_q', 'blocks.0.attn.hook_k', 'blocks.0.attn.hook_v', 'blocks.0.attn.hook_attn_scores', 'blocks.0.attn.hook_pattern', 'blocks.0.attn.hook_z', 'blocks.0.attn.hook_result', 'blocks.0.hook_attn_out', 'blocks.0.hook_resid_post', 'blocks.1.hook_resid_pre', 'blocks.1.attn.hook_q', 'blocks.1.attn.hook_k', 'blocks.1.attn.hook_v', 'blocks.1.attn.hook_attn_scores', 'blocks.1.attn.hook_pattern', 'blocks.1.attn.hook_z', 'blocks.1.attn.hook_result', 'blocks.1.hook_attn_out', 'blocks.1.hook_resid_post'])

In [6]:
import circuitsvis as cv
from IPython.display import display

str_tokens = model.to_str_tokens(text)
for layer in range(model.cfg.n_layers):
    attention_pattern = cache["pattern", layer]
    display(
        cv.attention.attention_patterns(tokens=str_tokens, attention=attention_pattern)
    )

In [7]:
def current_attn_detector(cache: tl.ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be current-token heads
    """
    attn_heads = []
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            attention_pattern = cache["pattern", layer][head]
            # take avg of diagonal elements
            score = attention_pattern.diagonal().mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


def prev_attn_detector(cache: tl.ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be prev-token heads
    """
    attn_heads = []
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            attention_pattern = cache["pattern", layer][head]
            # take avg of sub-diagonal elements
            score = attention_pattern.diagonal(-1).mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


def first_attn_detector(cache: tl.ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be first-token heads
    """
    attn_heads = []
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            attention_pattern = cache["pattern", layer][head]
            # take avg of 0th elements
            score = attention_pattern[:, 0].mean()
            if score > 0.4:
                attn_heads.append(f"{layer}.{head}")
    return attn_heads


print("Heads attending to current token  = ", ", ".join(current_attn_detector(cache)))
print("Heads attending to previous token = ", ", ".join(prev_attn_detector(cache)))
print("Heads attending to first token    = ", ", ".join(first_attn_detector(cache)))

Heads attending to current token  =  0.9
Heads attending to previous token =  0.7
Heads attending to first token    =  0.0, 0.1, 0.2, 0.3, 0.6, 0.8, 0.11, 1.0, 1.1, 1.3, 1.4, 1.10


In [8]:
from jaxtyping import Int, Float
from torch import Tensor


def generate_repeated_tokens(
    model: tl.HookedTransformer, seq_len: int, batch_size: int = 1
) -> Int[Tensor, "batch_size full_seq_len"]:
    """
    Generates a sequence of repeated random tokens

    Outputs are:
        rep_tokens: [batch_size, 1+2*seq_len]
    """
    t.manual_seed(0)  # for reproducibility
    prefix = (t.ones(batch_size, 1) * model.tokenizer.bos_token_id).long()
    rep_tokens_half = t.randint(
        0, model.cfg.d_vocab, (batch_size, seq_len), dtype=t.int64
    )
    rep_tokens = t.cat([prefix, rep_tokens_half, rep_tokens_half], dim=-1).to(device)
    return rep_tokens


def run_and_cache_model_repeated_tokens(
    model: tl.HookedTransformer, seq_len: int, batch_size: int = 1
) -> tuple[Tensor, Tensor, tl.ActivationCache]:
    """
    Generates a sequence of repeated random tokens, and runs the model on it, returning (tokens, logits, cache). This
    function should use the `generate_repeated_tokens` function above

    Outputs are:
        rep_tokens: [batch_size, 1+2*seq_len]
        rep_logits: [batch_size, 1+2*seq_len, d_vocab]
        rep_cache: The cache of the model run on rep_tokens
    """
    rep_tokens = generate_repeated_tokens(model, seq_len, batch_size)
    rep_logits, rep_cache = model.run_with_cache(rep_tokens)
    return rep_tokens, rep_logits, rep_cache


def get_log_probs(
    logits: Float[Tensor, "batch posn d_vocab"], tokens: Int[Tensor, "batch posn"]
) -> Float[Tensor, "batch posn-1"]:
    logprobs = logits.log_softmax(dim=-1)
    # We want to get logprobs[b, s, tokens[b, s+1]], in eindex syntax this looks like:
    correct_logprobs = eindex(logprobs, tokens, "b s [b s+1]")
    return correct_logprobs


from eindex import eindex

seq_len = 50
batch_size = 1
(rep_tokens, rep_logits, rep_cache) = run_and_cache_model_repeated_tokens(
    model, seq_len, batch_size
)
rep_cache.remove_batch_dim()
rep_str = model.to_str_tokens(rep_tokens)
model.reset_hooks()
log_probs = get_log_probs(rep_logits, rep_tokens).squeeze()

print(f"Performance on the first half: {log_probs[:seq_len].mean():.3f}")
print(f"Performance on the second half: {log_probs[seq_len:].mean():.3f}")


Performance on the first half: -14.923
Performance on the second half: -6.327


In [9]:
prompt = "How are you in the name of the ocean? How are you in the name of the"

tokens = model.to_tokens(prompt).to(device)

attn_model = attn_model.to(device)

demo_logits = attn_model(tokens)
reference_logits = model(tokens)

demo_completion = model.tokenizer.decode(demo_logits[-1, -1].argmax())
reference_completion = model.tokenizer.decode(reference_logits[-1, -1].argmax())


In [10]:
t.linalg.norm(demo_logits[0] - reference_logits[0], ord=1, dim=-1)

tensor([0.0815, 0.0738, 0.0988, 0.0983, 0.0989, 0.0745, 0.1054, 0.0898, 0.0809,
        0.1089, 0.0914, 0.0828, 0.1019, 0.1035, 0.0973, 0.0805, 0.1104, 0.0910,
        0.0804], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)

In [11]:
demo_logits[0, -1].topk(k=5), reference_logits[0, -1].topk(k=5)

(torch.return_types.topk(
 values=tensor([8.5582, 6.6525, 6.4197, 5.8962, 5.7753], device='cuda:0',
        grad_fn=<TopkBackward0>),
 indices=tensor([12927,  6150,  1533,  6215,  7565], device='cuda:0')),
 torch.return_types.topk(
 values=tensor([8.5582, 6.6525, 6.4197, 5.8962, 5.7753], device='cuda:0',
        grad_fn=<TopkBackward0>),
 indices=tensor([12927,  6150,  1533,  6215,  7565], device='cuda:0')))